In [1]:
import torch
import torch.nn as nn
import math
import urllib.request


In [2]:
# ============================================================
# 1. Device
# ============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)


Device: cuda


In [3]:
# ============================================================
# 2. Load Tiny Shakespeare
# ============================================================

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

text = urllib.request.urlopen(url).read().decode("utf-8")

print("Characters:", len(text))


Characters: 1115394


In [4]:
# ============================================================
# 3. Character vocabulary
# ============================================================

chars = sorted(set(text))

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

VOCAB_SIZE = len(chars)

print("Vocabulary:", VOCAB_SIZE)


# Convert text → integers
data = torch.tensor(
    [stoi[ch] for ch in text],
    dtype=torch.long
)


Vocabulary: 65


In [5]:
# ============================================================
# 4. Training data
# ============================================================


BATCH_SIZE = 32
BLOCK_SIZE = 64


def get_batch():

    starts = torch.randint(
        0,
        len(data) - BLOCK_SIZE - 1,
        (BATCH_SIZE,)
    )

    x = torch.stack([
        data[i:i + BLOCK_SIZE]
        for i in starts
    ])

    y = torch.stack([
        data[i + 1:i + BLOCK_SIZE + 1]
        for i in starts
    ])

    return x.to(DEVICE), y.to(DEVICE)



In [6]:
# ============================================================
# 5. Positional Encoding
# ============================================================

class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len):

        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(
            max_len
        ).unsqueeze(1)

        div = torch.exp(
            torch.arange(0, d_model, 2)
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div)

        pe[:, 1::2] = torch.cos(position * div)

        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x):

        T = x.shape[1]

        return x + self.pe[:, :T]



In [7]:
# ============================================================
# 6. Multi-Head Self Attention
# ============================================================

class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, num_heads):

        super().__init__()

        assert d_model % num_heads == 0

        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)

        self.Wo = nn.Linear(d_model, d_model)

        # Causal mask
        mask = torch.tril(
            torch.ones(BLOCK_SIZE, BLOCK_SIZE)
        )

        self.register_buffer(
            "mask",
            mask
        )

    def forward(self, x):

        B, T, C = x.shape

        # --------------------------------
        # Q K V
        # --------------------------------

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        # [B,T,C]
        #
        # → [B,T,H,D]
        #
        # → [B,H,T,D]

        Q = Q.view(
            B, T,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        K = K.view(
            B, T,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        V = V.view(
            B, T,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        # --------------------------------
        # Attention scores
        # --------------------------------

        scores = Q @ K.transpose(-2, -1)

        scores = scores / math.sqrt(
            self.head_dim
        )

        # --------------------------------
        # Causal masking
        # --------------------------------

        scores = scores.masked_fill(
            self.mask[:T, :T] == 0,
            float("-inf")
        )

        # --------------------------------
        # Softmax
        # --------------------------------

        weights = torch.softmax(
            scores,
            dim=-1
        )

        # --------------------------------
        # Weighted values
        # --------------------------------

        out = weights @ V

        # [B,H,T,D]
        #
        # → [B,T,H,D]

        out = out.transpose(1, 2)

        # Merge heads
        out = out.contiguous().view(
            B, T, C
        )

        # Output projection
        out = self.Wo(out)

        return out


In [8]:
# ============================================================
# 7. MLP
# ============================================================

class MLP(nn.Module):

    def __init__(self, d_model):

        super().__init__()

        self.fc1 = nn.Linear(
            d_model,
            4 * d_model
        )

        self.fc2 = nn.Linear(
            4 * d_model,
            d_model
        )

    def forward(self, x):

        x = self.fc1(x)

        x = torch.relu(x)

        x = self.fc2(x)

        return x


In [9]:
# ============================================================
# 8. Transformer Block
# ============================================================

class TransformerBlock(nn.Module):

    def __init__(self, d_model, num_heads):

        super().__init__()

        self.ln1 = nn.LayerNorm(d_model)

        self.attention = MultiHeadAttention(
            d_model,
            num_heads
        )

        self.ln2 = nn.LayerNorm(d_model)

        self.mlp = MLP(d_model)

    def forward(self, x):

        # Attention + residual
        x = x + self.attention(
            self.ln1(x)
        )

        # MLP + residual
        x = x + self.mlp(
            self.ln2(x)
        )

        return x



In [10]:
# ============================================================
# 9. GPT Model
# ============================================================

class GPT(nn.Module):

    def __init__(self):

        super().__init__()

        D_MODEL = 128
        NUM_HEADS = 4
        NUM_BLOCKS = 4

        # Character embedding
        self.embedding = nn.Embedding(
            VOCAB_SIZE,
            D_MODEL
        )

        # Positional encoding
        self.position = PositionalEncoding(
            D_MODEL,
            BLOCK_SIZE
        )

        # Transformer blocks
        self.blocks = nn.Sequential(
            *[
                TransformerBlock(
                    D_MODEL,
                    NUM_HEADS
                )
                for _ in range(NUM_BLOCKS)
            ]
        )

        # Final LayerNorm
        self.ln = nn.LayerNorm(D_MODEL)

        # Output vocabulary
        self.fc = nn.Linear(
            D_MODEL,
            VOCAB_SIZE
        )

    def forward(self, x, targets=None):

        # --------------------------------
        # Character → embedding
        # --------------------------------

        x = self.embedding(x)

        # [B,T] → [B,T,128]

        # --------------------------------
        # Add position
        # --------------------------------

        x = self.position(x)

        # --------------------------------
        # Transformer blocks
        # --------------------------------

        x = self.blocks(x)

        # --------------------------------
        # Final normalization
        # --------------------------------

        x = self.ln(x)

        # --------------------------------
        # Vocabulary logits
        # --------------------------------

        logits = self.fc(x)

        # [B,T,128]
        #
        # → [B,T,VOCAB_SIZE]

        # --------------------------------
        # Loss
        # --------------------------------

        loss = None

        if targets is not None:

            B, T, C = logits.shape

            logits = logits.view(
                B * T,
                C
            )

            targets = targets.view(
                B * T
            )

            loss = nn.functional.cross_entropy(
                logits,
                targets
            )

        return logits, loss


In [11]:
# ============================================================
# 10. Create model
# ============================================================

model = GPT().to(DEVICE)

print(
    "Parameters:",
    sum(p.numel() for p in model.parameters())
)


Parameters: 810049


In [12]:
# ============================================================
# 11. Optimizer
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)


In [16]:
# ============================================================
# 12. Training
# ============================================================

STEPS = 3000

model.train()

for step in range(STEPS):

    X, Y = get_batch()


    logits, loss = model(
        X,
        Y
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if step % 100 == 0:

        print(
            f"step {step}: "
            f"loss {loss.item():.4f}"
        )



step 0: loss 1.5447
step 100: loss 1.5293
step 200: loss 1.4939
step 300: loss 1.5197
step 400: loss 1.4946
step 500: loss 1.5156
step 600: loss 1.4642
step 700: loss 1.5216
step 800: loss 1.4926
step 900: loss 1.4880
step 1000: loss 1.4959
step 1100: loss 1.5386
step 1200: loss 1.4940
step 1300: loss 1.4617
step 1400: loss 1.5072
step 1500: loss 1.4027
step 1600: loss 1.4207
step 1700: loss 1.4938
step 1800: loss 1.4426
step 1900: loss 1.4179
step 2000: loss 1.5083
step 2100: loss 1.3528
step 2200: loss 1.4521
step 2300: loss 1.3948
step 2400: loss 1.4784
step 2500: loss 1.4025
step 2600: loss 1.3829
step 2700: loss 1.4706
step 2800: loss 1.5125
step 2900: loss 1.3959


In [17]:
# ============================================================
# 13. Generation
# ============================================================

@torch.no_grad()
def generate(
    prompt,
    max_new_tokens=300
):

    model.eval()

    # Prompt → token IDs

    x = torch.tensor(
        [[stoi[ch] for ch in prompt]],
        dtype=torch.long,
        device=DEVICE
    )

    for _ in range(max_new_tokens):

        # Keep only context window

        x_context = x[:, -BLOCK_SIZE:]

        # Prediction

        logits, _ = model(
            x_context
        )

        # Last character prediction

        logits = logits[:, -1, :]

        # Convert logits → probabilities

        probs = torch.softmax(
            logits,
            dim=-1
        )

        # Sample next character

        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        # Add to sequence

        x = torch.cat(
            [x, next_token],
            dim=1
        )

    # IDs → characters

    result = "".join(
        itos[i.item()]
        for i in x[0]
    )

    return result

In [19]:
# ============================================================
# 14. Generate
# ============================================================

print("\nGenerated:\n")

print(
    generate(
        "VINCENTIO:",
        500
    )
)


Generated:

VINCENTIO:
So what at young traitor with to hence!
The strew with his estate.

LUCIO:
O' conclaster, how his bright
That sall wrought you. How now thief to that it thee so.
There for not in Englands, you know head so been
it-matter, making less-silly, 'vengious wear fair.

LEONTES:
A laugh, Auck'd, good heread.
When his snot please thee deliver.

PETRUCHIO:

This, sir.

TYRREL:
How now, fellowed by Lucenter children'd hath within.

BAWHARWIAR Petrust:
Conjury, in him with a mightmore; go, a nothing;
The g
